In [1]:
# Cleaning
# Reads: the 3 raw files in ../data/ Saves: clean_life.csv, clean_mort.csv, clean_gdp.csv in ../data/

import pandas as pd
import numpy as np

In [2]:
# Load the raw files (never edited, so I can always rerun this notebook)
life_raw = pd.read_csv(r"C:\Users\admin\Desktop\Captoneproject-pydata\data\life-expectancy.csv")
mort_raw = pd.read_csv(r"C:\Users\admin\Desktop\Captoneproject-pydata\data\child-mortality.csv")
gdp_raw = pd.read_csv(r"C:\Users\admin\Desktop\Captoneproject-pydata\data\gdp-per-capita-maddison-project-database.csv")

print("Raw shapes:", life_raw.shape, mort_raw.shape, gdp_raw.shape)


Raw shapes: (21565, 4) (17066, 4) (21586, 5)


In [3]:
# renaming  the long value columns to short names
life = life_raw.rename(columns={"Life expectancy": "life_exp"})
mort = mort_raw.rename(columns={"Under-five mortality rate (selected)": "child_mort"})
gdp = gdp_raw.rename(columns={"GDP per capita": "gdp_pc"})


In [4]:
# dropping the annotations column in the GDP file like almost empty, not needed)
print("\nGDP annotations non-empty rows:", gdp["GDP per capita (Annotations)"].notna().sum())
gdp = gdp.drop(columns=["GDP per capita (Annotations)"])



GDP annotations non-empty rows: 28


In [5]:
## types and text tidy up
for name, d in [("life", life), ("mort", mort), ("gdp", gdp)]:
    d["Entity"] = d["Entity"].astype(str).str.strip()
    d["Code"] = d["Code"].astype("string").str.strip().str.upper()
    d["Year"] = d["Year"].astype(int)
print("\nTypes after tidy up:")
print(life.dtypes)


Types after tidy up:
Entity          str
Code         string
Year          int64
life_exp    float64
dtype: object


In [6]:
# missing values in the columns I need
print("\nMissing values before cleaning")
for name, d, col in [("life", life, "life_exp"), ("mort", mort, "child_mort"), ("gdp", gdp, "gdp_pc")]:
    print(f"  {name}: value column {d[col].isna().sum()}, Entity {d['Entity'].isna().sum()}, Year {d['Year'].isna().sum()}")



Missing values before cleaning
  life: value column 0, Entity 0, Year 0
  mort: value column 0, Entity 0, Year 0
  gdp: value column 0, Entity 0, Year 0


In [7]:
##Bemove aggregates (regions, groups, historic states with no modern country code)
def is_aggregate(d):
    return d["Code"].isna() | d["Code"].astype(str).str.startswith("OWID")

removed = {}
for name, d in [("life", life), ("mort", mort), ("gdp", gdp)]:
    removed[name] = sorted(d.loc[is_aggregate(d), "Entity"].unique())

life_c = life[~is_aggregate(life)].copy()
mort_c = mort[~is_aggregate(mort)].copy()
gdp_c = gdp[~is_aggregate(gdp)].copy()

print("\nRows removed as aggregates:")
print("  life:", len(life) - len(life_c))
print("  mort:", len(mort) - len(mort_c))
print("  gdp: ", len(gdp) - len(gdp_c))
print("\nExamples removed from GDP:", removed["gdp"][:12])
print("Examples removed from child mortality:", removed["mort"][:12])



Rows removed as aggregates:
  life: 2113
  mort: 905
  gdp:  652

Examples removed from GDP: ['Czechoslovakia', 'East Asia (Maddison)', 'Eastern Europe (Maddison)', 'Latin America (Maddison)', 'Middle East and North Africa (Maddison)', 'South and South East Asia (Maddison)', 'Sub Saharan Africa (Maddison)', 'Sudan (former)', 'USSR', 'Western Europe (Maddison)', 'Western offshoots (Maddison)', 'World']
Examples removed from child mortality: ['Africa', 'Asia', 'Europe', 'European Union (27)', 'High-income countries', 'Kosovo', 'Low-income countries', 'Lower-middle-income countries', 'North America', 'Oceania', 'South America', 'Upper-middle-income countries']


In [8]:
# drop rows with a missing value in the column I need (none expected)
life_c = life_c.dropna(subset=["life_exp"])
mort_c = mort_c.dropna(subset=["child_mort"])
gdp_c = gdp_c.dropna(subset=["gdp_pc"])


In [9]:
# duplicates on the join keys
for name, d in [("life", life_c), ("mort", mort_c), ("gdp", gdp_c)]:
    n_dup = d.duplicated(["Entity", "Code", "Year"]).sum()
    print(f"Duplicate Entity+Code+Year rows in {name}: {n_dup}")
    d.drop_duplicates(["Entity", "Code", "Year"], inplace=True)


Duplicate Entity+Code+Year rows in life: 0
Duplicate Entity+Code+Year rows in mort: 0
Duplicate Entity+Code+Year rows in gdp: 0


In [10]:
# impossible values
print("\nValue ranges (should be sensible)")
print("  life_exp:  ", life_c["life_exp"].min(), "to", life_c["life_exp"].max(), "(years)")
print("  child_mort:", round(mort_c["child_mort"].min(), 2), "to", round(mort_c["child_mort"].max(), 2))
print("  gdp_pc:    ", round(gdp_c["gdp_pc"].min()), "to", round(gdp_c["gdp_pc"].max()), "(international dollars)")
print("  Any life_exp outside 0 to 120?", ((life_c["life_exp"] <= 0) | (life_c["life_exp"] > 120)).sum())
print("  Any GDP at or below 0?", (gdp_c["gdp_pc"] <= 0).sum())
print("  Any child_mort below 0 or above 100?", ((mort_c["child_mort"] < 0) | (mort_c["child_mort"] > 100)).sum())



Value ranges (should be sensible)
  life_exp:   10.9891 to 86.3724 (years)
  child_mort: 0.13 to 78.09
  gdp_pc:     295 to 160051 (international dollars)
  Any life_exp outside 0 to 120? 0
  Any GDP at or below 0? 0
  Any child_mort below 0 or above 100? 0


In [11]:
# do the country names and codes agree across files?
codes = {"life": set(life_c["Code"]), "mort": set(mort_c["Code"]), "gdp": set(gdp_c["Code"])}
print("\nCountry codes per file:", {k: len(v) for k, v in codes.items()})
print("In all three:", len(codes["life"] & codes["mort"] & codes["gdp"]))
print("In life but not GDP:", sorted(codes["life"] - codes["gdp"])[:15])
print("In GDP but not life:", sorted(codes["gdp"] - codes["life"])[:15])
print("In mort but not GDP:", sorted(codes["mort"] - codes["gdp"])[:15])

# Does one code always map to one country name across files?
names = pd.concat([life_c[["Code", "Entity"]], mort_c[["Code", "Entity"]], gdp_c[["Code", "Entity"]]]).drop_duplicates()
clash = names.groupby("Code")["Entity"].nunique()
print("Codes with more than one country name:", int((clash > 1).sum()))
if (clash > 1).sum() > 0:
    print(names[names["Code"].isin(clash[clash > 1].index)].sort_values("Code"))



Country codes per file: {'life': 236, 'mort': 200, 'gdp': 165}
In all three: 163
In life but not GDP: ['ABW', 'AIA', 'AND', 'ASM', 'ATG', 'BES', 'BHS', 'BLM', 'BLZ', 'BMU', 'BRN', 'BTN', 'COK', 'CUW', 'CYM']
In GDP but not life: []
In mort but not GDP: ['AIA', 'AND', 'ATG', 'BHS', 'BLZ', 'BRN', 'BTN', 'COK', 'ERI', 'FJI', 'FSM', 'GRD', 'GUY', 'KIR', 'KNA']
Codes with more than one country name: 0


In [12]:
# before and after row counts
summary_table = pd.DataFrame({
    "Raw rows": [len(life_raw), len(mort_raw), len(gdp_raw)],
    "Clean rows": [len(life_c), len(mort_c), len(gdp_c)],
}, index=["life expectancy", "child mortality", "GDP per capita"])
summary_table["Rows removed"] = summary_table["Raw rows"] - summary_table["Clean rows"]
print("\nBEFORE AND AFTER")
display(summary_table)



BEFORE AND AFTER


,Raw rows,Clean rows,Rows removed
life expectancy,21565,19452,2113
child mortality,17066,16161,905
GDP per capita,21586,20934,652


In [13]:
#
#save the cleaned files...............(raw files stay untouched)

life_c.to_csv("../data/clean_life.csv", index=False)
mort_c.to_csv("../data/clean_mort.csv", index=False)
gdp_c.to_csv("../data/clean_gdp.csv", index=False)
print("\nSaved: clean_life.csv, clean_mort.csv, clean_gdp.csv in the data folder")



Saved: clean_life.csv, clean_mort.csv, clean_gdp.csv in the data folder


In [14]:
# Final check by reloading one file
check = pd.read_csv("../data/clean_life.csv")
print("Reloaded clean_life.csv:", check.shape, list(check.columns))

Reloaded clean_life.csv: (19452, 4) ['Entity', 'Code', 'Year', 'life_exp']
